In [57]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np

In [58]:
df = pd.read_csv("powerplant_data.csv")

In [59]:
df.head()
#PE is the output/label.
#AT is temperature value
#V is vacccume value
#AP is pressure value
#RH is humidity value
#PE is produced energy value

,AT,V,AP,RH,PE
0,8.34,40.77,1010.84,90.01,480.48
1,23.64,58.49,1011.40,74.20,445.75
2,29.74,56.90,1007.15,41.91,438.76
3,19.07,49.69,1007.22,76.79,453.09
4,11.80,40.66,1017.13,97.20,464.43


In [60]:
df.isnull().sum()

AT    0
V     0
AP    0
RH    0
PE    0
dtype: int64

In [61]:
X = df.drop(columns=["PE"])
Y = df["PE"]

In [62]:
# Split our data
from sklearn.model_selection import train_test_split
X_train, X_test, Y_train, Y_test = train_test_split(
    X, Y, test_size=0.2, random_state=42
)

In [63]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Coverting data to tensors

In [64]:
import torch
import torch.nn as nn

In [65]:
X_train_tensor = torch.tensor(X_train_scaled, dtype=torch.float32)
Y_train_tensor = torch.tensor(Y_train.values, dtype=torch.float32).view(-1,1)

X_test_tensor = torch.tensor(X_test_scaled, dtype=torch.float32)
Y_test_tensor = torch.tensor(Y_test.values, dtype=torch.float32).view(-1,1)

In [66]:
type(X_train_scaled)

numpy.ndarray

In [67]:
type(Y_train)
#as you can see this is not a numpy array. its a pandas series. so need to use .values.
#now .view changes dimension/shape.


pandas.core.series.Series

In [68]:
# Some concept before proceeding
#so df ko humne tensors ke andar convert kiya hai.
#tensors ram ke andar store hai.
#DataLoader class
#suppose in tensors you have 1000 datapoints.
#you have to create batches with which youll send data to nn.
#that is done via DataLoader and can also shuffle the samples within batches.
#Now DataLoader cannot directly access memory data.
#So we use a helper Class called TensorDataset class.
#it picks raw data and gives it to DataLoader.

In [69]:
from torch.utils.data import TensorDataset, DataLoader

#train_dataset = TensorDataset(in_features, out_features)
train_dataset = TensorDataset(X_train_tensor, Y_train_tensor)
#test_dataset = TensorDataset(in_features, out_features)
test_dataset = TensorDataset(X_test_tensor, Y_test_tensor)

In [70]:
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32)

# Building and training Ann model.

In [71]:
#Deep-learning
#Defining ann model
#basically forward prop
#isko manually likhna padta hai
#but backprop apne aap ho jata hai
class ANN(nn.Module):
    def __init__(self):
        super(ANN, self).__init__()
        
        self.model = nn.Sequential(
            #1st hidden layer
            #nn.linear(in_features, out_features)
            nn.Linear(X_train.shape[1], 6),
            nn.ReLU(),

            #2nd hidden layer
            nn.Linear(6,6),
            nn.ReLU(),

            #output-layer
            nn.Linear(6,1)
        )

    def forward(self, x):
        return self.model(x)

In [72]:
#Building ann model
import torch.optim as optim

model = ANN()

#loss, optimizer
crietrion = nn.MSELoss() #loss-func
optimizer = optim.Adam(model.parameters()) #parameters here means weights and bias.

In [73]:
#Training/learning of ann
#weight updates so that produce right output
train_losses = []
val_losses = []

best_val_loss = float("inf")

epochs = 100

for epoch in range(epochs):
    model.train() #training-mode #here our model knows ke use weights ko adjust karna hai.
    running_loss = 0.0 #save the total training loss/error for one epoch.

    for xb, yb in train_loader:
        #xb = feartures of one single batch
        #yb = labels of one single batch
        #ese karke training dataset ke saare batches ke upar train karenge. ek batch pass hoga at a given time.
        optimizer.zero_grad() #purane gradiants wapas se na repeat ho in new batch training.

        outputs = model(xb) #predicted outputs for this batch #FP
        loss = crietrion(outputs, yb) #compute loss/error for this output. #ismein outputs and actual labels ko pass kiya.
        loss.backward() #back prop.. compute gradients #BP
        optimizer.step() #params(weight/bias) update
        running_loss += loss.item() #loss is a tensor value so converting to python float via .item()
    #now saare batches ka running loss aa gya for 1 epoch.
    epoch_train_loss = running_loss/len(train_loader) #no.of batches len(train_loader) se ayega.
    train_losses.append(epoch_train_loss)

    #saath saath har ek epoch ke saath testing data pe validate bhi kar rahe hain.
    #Validation
    model.eval() #evaluation mode
    running_val_loss = 0.0

    with torch.no_grad(): #specificaly telling to not calculate gradients #for fast computation and save memory.
        for xb, yb in test_loader:
            outputs = model(xb)
            loss = crietrion(outputs, yb)
            #here no backprop #no need to re-adjust weight.
            #valdiation simply means training ke time par kuch dataset ke saath validate kar lete hai 
            #taaki we are assured ke model sahi se kaam kare
            running_val_loss += loss

    epoch_val_loss = running_val_loss/len(test_loader)
    val_losses.append(epoch_val_loss)

    print(f"epoch {epoch+1}/{epochs} ==> training loss = ${epoch_train_loss} & val loss = ${epoch_val_loss}")
    if epoch_val_loss < best_val_loss:
        best_val_loss = epoch_val_loss
        torch.save(model.state_dict(), "best_model.pt")

epoch 1/100 ==> training loss = $205522.00286458334 & val loss = $202387.234375
epoch 2/100 ==> training loss = $190780.87102864584 & val loss = $172113.203125
epoch 3/100 ==> training loss = $142869.33570963543 & val loss = $109675.34375
epoch 4/100 ==> training loss = $79118.41285807292 & val loss = $52899.75390625
epoch 5/100 ==> training loss = $38224.44581705729 & val loss = $27829.09765625
epoch 6/100 ==> training loss = $22923.60061035156 & val loss = $19180.759765625
epoch 7/100 ==> training loss = $16782.448494466145 & val loss = $14648.0888671875
epoch 8/100 ==> training loss = $12844.178076171875 & val loss = $11032.4404296875
epoch 9/100 ==> training loss = $9380.243412272135 & val loss = $7839.2705078125
epoch 10/100 ==> training loss = $6456.499812825521 & val loss = $5185.71630859375
epoch 11/100 ==> training loss = $4136.707391866048 & val loss = $3236.049072265625
epoch 12/100 ==> training loss = $2595.2611923217773 & val loss = $2083.350341796875
epoch 13/100 ==> trai

In [74]:
#Loading best model
model.load_state_dict(torch.load("best_model.pt"))

<All keys matched successfully>

In [75]:
#Evaluate our model

model.eval()
with torch.no_grad():
    Y_train_pred = model(X_train_tensor)
    Y_test_pred = model(X_test_tensor)

    train_mse_loss = crietrion(Y_train_pred, Y_train_tensor)
    test_mse_loss = crietrion(Y_test_pred, Y_test_tensor)

print("Training MSE:", train_mse_loss)
print("Test MSE:", test_mse_loss)

Training MSE: tensor(20.6868)
Test MSE: tensor(18.8761)


In [76]:
from sklearn.metrics import r2_score

print("r^2 score =", r2_score(Y_test, Y_test_pred))

r^2 score = 0.9340327859524957


# Comparing with ML models

In [88]:
#svr
from sklearn.svm import SVR

In [89]:
model_svr = SVR()
model_svr.fit(X_train_scaled, Y_train)

,"kernel kernel: {'linear', 'poly', 'rbf', 'sigmoid', 'precomputed'} or callable, default='rbf'Specifies the kernel type to be used in the algorithm.If none is given, 'rbf' will be used. If a callable is given it isused to precompute the kernel matrix.For an intuitive visualization of different kernel typessee :ref:`sphx_glr_auto_examples_svm_plot_svm_regression.py`",'rbf'
,"degree degree: int, default=3Degree of the polynomial kernel function ('poly').Must be non-negative. Ignored by all other kernels.",3
,"gamma gamma: {'scale', 'auto'} or float, default='scale'Kernel coefficient for 'rbf', 'poly' and 'sigmoid'.- if ``gamma='scale'`` (default) is passed then it uses 1 / (n_features * X.var()) as value of gamma,- if 'auto', uses 1 / n_features- if float, must be non-negative... versionchanged:: 0.22 The default value of ``gamma`` changed from 'auto' to 'scale'.",'scale'
,"coef0 coef0: float, default=0.0Independent term in kernel function.It is only significant in 'poly' and 'sigmoid'.",0.0
,"tol tol: float, default=1e-3Tolerance for stopping criterion.",0.001
,"C C: float, default=1.0Regularization parameter. The strength of the regularization isinversely proportional to C. Must be strictly positive.The penalty is a squared l2. For an intuitive visualization of theeffects of scaling the regularization parameter C, see:ref:`sphx_glr_auto_examples_svm_plot_svm_scale_c.py`.",1.0
,"epsilon epsilon: float, default=0.1Epsilon in the epsilon-SVR model. It specifies the epsilon-tubewithin which no penalty is associated in the training loss functionwith points predicted within a distance epsilon from the actualvalue. Must be non-negative.",0.1
,"shrinking shrinking: bool, default=TrueWhether to use the shrinking heuristic.See the :ref:`User Guide `.",True
,"cache_size cache_size: float, default=200Specify the size of the kernel cache (in MB).",200
,"verbose verbose: bool, default=FalseEnable verbose output. Note that this setting takes advantage of aper-process runtime setting in libsvm that, if enabled, may not workproperly in a multithreaded context.",False
,"max_iter max_iter: int, default=-1Hard limit on iterations within solver, or -1 for no limit.",-1


In [90]:
Y_pred = model_svr.predict(X_test_scaled)

In [91]:
print("r^2 score =", r2_score(Y_test, Y_pred))

r^2 score = 0.9432546600611768


In [92]:
#Random-forest-regressor

In [93]:
from sklearn.ensemble import RandomForestRegressor

In [94]:
model_forest = RandomForestRegressor(
    n_estimators=201,
    max_depth=3,
    oob_score=True,
    random_state=42
)

In [95]:
model_forest.fit(X_train_scaled, Y_train)

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",201
,"criterion criterion: {""squared_error"", ""absolute_error"", ""friedman_mse"", ""poisson""}, default=""squared_error""The function to measure the quality of a split. Supported criteriaare ""squared_error"" for the mean squared error, which is equal tovariance reduction as feature selection criterion and minimizes the L2loss using the mean of each terminal node, ""friedman_mse"", which usesmean squared error with Friedman's improvement score for potentialsplits, ""absolute_error"" for the mean absolute error, which minimizesthe L1 loss using the median of each terminal node, and ""poisson"" whichuses reduction in Poisson deviance to find splits.Training using ""absolute_error"" is significantly slowerthan when using ""squared_error""... versionadded:: 0.18 Mean Absolute Error (MAE) criterion... versionadded:: 1.0 Poisson criterion.",'squared_error'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",3
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=1.0The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None or 1.0, then `max_features=n_features`... note:: The default of 1.0 is equivalent to bagged trees and more randomness can be achieved by setting smaller values, e.g. 0.3... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to 1.0.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",1.0
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples a

In [96]:
Y_PRed = model_forest.predict(X_test_scaled)

In [97]:
print("r^2 score =", r2_score(Y_test, Y_pred))

r^2 score = 0.9432546600611768
